# 16. Out-of-fold stacking

One variable against ledger row 24: the same eighteen members, the same logistic
combiner, the same folds. **Only the combiner's fitting protocol changes.**

Row 24 fit the combiner on all 691,369 out-of-fold rows and scored it on those same
rows. The gain it claims is honest, because it was measured by fitting on half the
rows and scoring on the other half, but the CV in the ledger is an in-sample number
for the stacker and is not comparable to any other row in the file. That is a caveat
where a number belongs.

Here the combiner is fit inside the fold loop: for each of the five folds it is fit on
the other four and scored on the held-out one. The member vectors are already
out-of-fold, so a row's member predictions come from models that never saw it and its
combiner weights come from a fit that never saw it either.

The submission averages the five fold combiners, which is what every other submission
in this repo does with its five fold models.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


def find_repo():
    for b in [Path.cwd(), *Path.cwd().parents]:
        if (b / "data" / "raw" / "train.csv").exists():
            return b
    raise FileNotFoundError("data/raw/train.csv not found")


REPO = find_repo()
O, S = REPO / "artifacts" / "oof", REPO / "submissions"

train = pd.read_csv(REPO / "data" / "raw" / "train.csv")
test = pd.read_csv(REPO / "data" / "raw" / "test.csv")
y = train["addicted_label"].to_numpy()

# The same split every vector on disk was produced under. Rebuilt rather than loaded,
# and then checked, because a silently different fold vector is the one error here
# that produces a clean-looking wrong answer.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=42).split(train, y)):
    folds[va] = i
assert (folds >= 0).all() and np.bincount(folds).sum() == len(train)
print(f"train {len(train):,}  test {len(test):,}  folds {np.bincount(folds)}")

train 691,369  test 296,302  folds [138274 138274 138274 138274 138273]


In [2]:
# (name, out-of-fold vector, test predictions). The twelve raw-feature models were run
# before test vectors were being saved as .npy, so their test predictions come from
# the submission csvs, which is what those files are for.
MEM = [
    ("te42", O / "te_bag42_oof.npy", O / "te_bag42_test.npy"),
    ("te2024", O / "te_seed2024_oof.npy", O / "te_seed2024_test.npy"),
    ("te7", O / "te_seed7_oof.npy", O / "te_seed7_test.npy"),
    ("te2025", O / "te_seed2025_oof.npy", O / "te_seed2025_test.npy"),
    ("te13", O / "te_seed13_oof.npy", O / "te_seed13_test.npy"),
    ("anchor", O / "lgbm_default_anchor_seed42.npy",
     S / "lgbm_default_anchor_seed42.csv"),
    ("trees300", O / "lgbm_trees300_seed42.npy", S / "lgbm_trees300_seed42.csv"),
    ("trees1000", O / "lgbm_trees1000_seed42.npy", S / "lgbm_trees1000_seed42.csv"),
    ("trees2000", O / "lgbm_trees2000_seed42.npy", S / "lgbm_trees2000_seed42.csv"),
    ("lr010", O / "lgbm_lr01_n1000_seed42.npy", S / "lgbm_lr01_n1000_seed42.csv"),
    ("lr005", O / "lgbm_lr005_n2000_seed42.npy", S / "lgbm_lr005_n2000_seed42.csv"),
    ("lr003", O / "lgbm_lr003_n3333_seed42.npy", S / "lgbm_lr003_n3333_seed42.csv"),
    ("bag42", O / "lgbm_bag08_lr005_n2000_seed42.npy",
     S / "lgbm_bag08_lr005_n2000_seed42.csv"),
    ("bag2024", O / "lgbm_bag08_lr005_n2000_seed2024.npy",
     S / "lgbm_bag08_lr005_n2000_seed2024.csv"),
    ("bag7", O / "lgbm_bag08_lr005_n2000_seed7.npy",
     S / "lgbm_bag08_lr005_n2000_seed7.csv"),
    ("bag2025", O / "lgbm_bag08_lr005_n2000_seed2025.npy",
     S / "lgbm_bag08_lr005_n2000_seed2025.csv"),
    ("bag13", O / "lgbm_bag08_lr005_n2000_seed13.npy",
     S / "lgbm_bag08_lr005_n2000_seed13.csv"),
    ("neural", O / "neural_oof.npy", O / "neural_test.npy"),
]


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


def load_test(path):
    if path.suffix == ".npy":
        return np.load(path)
    df = pd.read_csv(path)
    # A csv written in a different row order would blend perfectly cleanly and be
    # undetectable in the score. Checked rather than assumed.
    assert (df["id"].to_numpy() == test["id"].to_numpy()).all(), f"id order {path.name}"
    return df["addicted_label"].to_numpy()


names = [m[0] for m in MEM]
Poof = {n: np.load(p) for n, p, _ in MEM}
Ptest = {n: load_test(t) for n, _, t in MEM}

for n in names:
    assert Poof[n].shape == (len(train),), n
    assert Ptest[n].shape == (len(test),), n
    # A partially failed run leaves a constant fold, which blends silently.
    assert min(np.ptp(Poof[n][folds == f]) for f in range(5)) > 0, f"dead fold in {n}"

Loof = np.column_stack([logit(Poof[n]) for n in names])
Ltest = np.column_stack([logit(Ptest[n]) for n in names])
print(f"{len(names)} members, oof {Loof.shape}, test {Ltest.shape}")
print("member CV:")
for n in names:
    cv = np.mean([roc_auc_score(y[folds == f], Poof[n][folds == f]) for f in range(5)])
    print(f"  {n:10} {cv:.6f}")

18 members, oof (691369, 18), test (296302, 18)
member CV:


  te42       0.966782


  te2024     0.966771


  te7        0.966729


  te2025     0.966743


...

  bag2024    0.963234


  bag7       0.963445


  bag2025    0.963337


  bag13      0.963483


  neural     0.939169


In [3]:
# The fold loop. Fit on four folds of the out-of-fold matrix, score on the fifth.
oof_stack = np.zeros(len(train))
test_by_fold = np.zeros((5, len(test)))
coefs = np.zeros((5, len(names)))

for f in range(5):
    tr, va = folds != f, folds == f
    clf = LogisticRegression(C=1.0, max_iter=2000).fit(Loof[tr], y[tr])
    oof_stack[va] = clf.decision_function(Loof[va])
    test_by_fold[f] = clf.decision_function(Ltest)
    coefs[f] = clf.coef_[0]
    print(f"  fold {f}: {roc_auc_score(y[va], oof_stack[va]):.6f}")

per_fold = np.array([roc_auc_score(y[folds == f], oof_stack[folds == f])
                     for f in range(5)])
CV, SD = per_fold.mean(), per_fold.std()
print(f"\nCV {CV:.6f} +/- {SD:.6f}")

  fold 0: 0.967003


  fold 1: 0.967801


  fold 2: 0.967974


  fold 3: 0.968179


  fold 4: 0.967292



CV 0.967650 +/- 0.000437


In [4]:
# Paired against the best single member on the identical folds, which is the
# comparison the ledger is read on.
base = np.array([roc_auc_score(y[folds == f], Poof["te42"][folds == f])
                 for f in range(5)])
d = per_fold - base
print(f"vs te42 (row 17)   {d.mean():+.6f}  paired sd {d.std(ddof=1):.6f}  "
      f"{(d > 0).sum()}/5 folds")
print("  per fold: " + "  ".join(f"{v:+.6f}" for v in d))

# And the number row 24 recorded, which is this same combiner scored on the rows it
# was fit on. The gap between the two is the optimism being removed.
insample = LogisticRegression(C=1.0, max_iter=2000).fit(Loof, y)
ins = np.array([roc_auc_score(y[folds == f],
                              insample.decision_function(Loof)[folds == f])
                for f in range(5)])
print(f"\nin-sample, as recorded in row 24: {ins.mean():.6f}")
print(f"out-of-fold, recorded here:       {CV:.6f}")
print(f"optimism removed:                 {ins.mean() - CV:+.6f}")

vs te42 (row 17)   +0.000867  paired sd 0.000051  5/5 folds
  per fold: +0.000894  +0.000825  +0.000801  +0.000916  +0.000902



in-sample, as recorded in row 24: 0.967665
out-of-fold, recorded here:       0.967650
optimism removed:                 +0.000015


In [5]:
# Coefficient stability. Five fits on 80% overlapping data should agree closely; if
# they did not, the combiner would be fitting fold noise and its weights would not
# mean anything.
print(f"{'member':10} {'mean':>9} {'sd across folds':>17} {'full fit':>10}")
order = np.argsort(-coefs.mean(axis=0))
for i in order:
    print(f"{names[i]:10} {coefs[:, i].mean():>+9.4f} {coefs[:, i].std():>17.4f} "
          f"{insample.coef_[0][i]:>+10.4f}")
print(f"\nlargest fold-to-fold sd: {coefs.std(axis=0).max():.4f}")

member          mean   sd across folds   full fit
te42         +0.1692            0.0064    +0.1699
te13         +0.1653            0.0070    +0.1662
te2024       +0.1648            0.0134    +0.1644
te2025       +0.1553            0.0131    +0.1551
te7          +0.1362            0.0165    +0.1371
lr003        +0.1271            0.0145    +0.1302
neural       +0.1178            0.0020    +0.1178
lr005        +0.1065            0.0084    +0.1097
bag42        +0.0932            0.0124    +0.0949
bag7         +0.0893            0.0194    +0.0906
bag13        +0.0844            0.0097    +0.0851
bag2025      +0.0687            0.0055    +0.0682
bag2024      +0.0197            0.0387    +0.0101
trees1000    -0.0076            0.0095    -0.0096
trees2000    -0.0126            0.0024    -0.0118
lr010        -0.0157            0.0070    -0.0195
trees300     -0.1349            0.0095    -0.1321
anchor       -0.3630            0.0049    -0.3619

largest fold-to-fold sd: 0.0387


In [6]:
# Submission: the average of the five fold combiners, matching what every other
# submission in this repo does with its five fold models.
pred = test_by_fold.mean(axis=0)
prob = 1 / (1 + np.exp(-pred))

# The alternative would be the single combiner fit on all out-of-fold rows. AUC reads
# only the ordering, so what matters is whether the two orderings differ at all.
alt = insample.decision_function(Ltest)
tau = pd.Series(pred).corr(pd.Series(alt), method="spearman")
print(f"spearman between the fold-averaged and full-fit test orderings: {tau:.8f}")

out = REPO / "submissions" / "stack_oof_18.csv"
sub = pd.DataFrame({"id": test["id"], "addicted_label": prob})
assert len(sub) == len(test) and sub["addicted_label"].between(0, 1).all()
sub.to_csv(out, index=False)
print(f"wrote {out.name}, {len(sub):,} rows, "
      f"range [{prob.min():.4f}, {prob.max():.4f}]")
print(f"\nledger: CV {CV:.6f} +/- {SD:.6f}, vs te42 {d.mean():+.6f} "
      f"({(d > 0).sum()}/5, sd {d.std(ddof=1):.6f})")

spearman between the fold-averaged and full-fit test orderings: 0.99999949


wrote stack_oof_18.csv, 296,302 rows, range [0.0001, 1.0000]

ledger: CV 0.967650 +/- 0.000437, vs te42 +0.000867 (5/5, sd 0.000051)
